Looking at the files generated

### Top level view of the generated files and folders: Counts

In [1]:
import json
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 60)
DATA = Path("data")

rows = []
for folder in sorted(p for p in DATA.rglob("*") if p.is_dir()):
    files = [f for f in folder.rglob("*") if f.is_file()]
    rows.append({"folder": str(folder.relative_to(DATA)), "files": len(files),
                 "MB": round(sum(f.stat().st_size for f in files) / 1e6, 1)})
pd.DataFrame(rows).query("folder.str.count('/') < 2")

,folder,files,MB
0,biotools,3,0.1
1,download,3282,792.2
2,download/crates,787,736.6
3,download/ga,876,37.1
791,download/meta,1617,18.3
792,galaxy,3,0.6
793,mapping,2,3.9


The 1617 files in download/meta tells us that we sucessfully downloaded 1617/1620 workflows from workflowhub. I got a couple of errors while running so that makes sense. 
The 787 files in download/crates represent all uploaded workflows that contain ga files (some of which have multple) and download/ga have the final number of ga workflows extracted from these crates: 876. 


In [2]:
index = json.loads((DATA / "download/index.json").read_text())
wf = pd.read_csv(DATA / "download/workflows.csv", dtype={"id": str})
print("rows in workflows.csv:", len(wf))

rows in workflows.csv: 1617


The index contains notes of all 1620 workflows on workflowhub. IDs are automatically generated - they start at 2 and go to 2374, so they might not look like IDs but they are.
Workflows.csv contains the 1617 successful downloads, which matches download/meta - makes sense

In [3]:
wf["workflow_class"].value_counts()

workflow_class
Galaxy                                      788
Nextflow                                    212
Common Workflow Language                    123
Snakemake                                    83
COMP Superscalar                             75
Python                                       66
Unrecognized workflow type                   47
Jupyter                                      46
Workflow Description Language                33
Docker                                       27
Scipion                                      20
Shell Script                                 19
KNIME                                        15
Apache Airflow                               15
R markdown                                   10
Autosubmit                                   10
Playbook Workflow Builder Workflow            9
Bpipe                                         5
Quarto Markdown                               2
BioCompute Object                             2
Argo Workflow            

It says 788 galaxy workflows, so probably 1 of the errors we see in the 1617/1620 was a galaxy workflow. 

### Checking which ones have multiple files and which ones have none


In [4]:
galaxy = wf[wf["workflow_class"].str.lower() == "galaxy"].copy()
galaxy["has_ga"] = galaxy["ga_files"].notna()
galaxy["n_ga"] = galaxy["ga_files"].fillna("").str.split(";").map(lambda x: len([g for g in x if g]))
print(galaxy["has_ga"].value_counts())
print("\nGalaxy workflows with more than one .ga:", (galaxy["n_ga"] > 1).sum())
galaxy.loc[~galaxy["has_ga"], ["id", "title"]]

has_ga
True     779
False      9
Name: count, dtype: int64

Galaxy workflows with more than one .ga: 31


,id,title
458,685,mRNA-Seq BY-COVID Pipeline
697,1113,Full Analyse Argo data
805,1314,Galaxy Workflow for CNV Detection with CNVkit and Conver...
1396,2083,Example deep metabolome annotation LC-MS/MS data process...
1397,2084,Deep metabolome annotation LC-MS/MS fractionation and DI...
1398,2085,Deep metabolome annotation LC-MS/MS workflow (W2)
1399,2086,Deep metabolome annotation LC-MS/MS (metabolite standard...
1464,2167,GlyCombo Galaxy
1596,2350,AOPTK download and analysis


Numbers add up! TODO: perhaps we should look into/exclude the ones that have multiples - could be same workflow repeated, for example the ones wth multiple verisons

In [ ]:
# Looking at most common topics for galaxy workflows - we are only interested in life science ones
galaxy["topics"].dropna().str.split(";").explode().value_counts().head(20)

topics
Whole genome sequencing    15
Sequence assembly          14
Ecology                    12
Biodiversity               10
Genetic variation           8
Transcriptomics             8
Biomedical science          8
Synthetic biology           8
Population genomics         7
Gene expression             6
Bioinformatics              6
Genomics                    6
Metagenomics                5
Virology                    5
Bioimaging                  4
Imaging                     4
Microbiology                4
Physics                     4
Metabolomics                4
Oncology                    3
Name: count, dtype: int64

### Looking at edges extracted by `galaxy_tool_pairs.py`

In [24]:
pairs = pd.read_csv(DATA / "galaxy/galaxy_pairs.csv", dtype={"workflow_id": str})
gedges = pd.read_csv(DATA / "galaxy/galaxy_edges.csv")
gtools = pd.read_csv(DATA / "galaxy/galaxy_tools.csv")
print("Nr connections:", len(pairs), ", nr unique connections:", len(gedges), f", nr tools:", len(gtools))
print("workflows with at least one pair:", pairs["workflow_id"].nunique(), "of", galaxy["has_ga"].sum())
print("self-loops", (pairs["source"] == pairs["target"]).sum())

Nr connections: 5091 , nr unique connections: 2182 , nr tools: 1186
workflows with at least one pair: 629 of 779
self-loops 0


So 150 workflows out of 779 did not have any identifiable edges. Should check which of these were, it might just be that they simply had input -> tool -> output, in which case the code would not collect any edges.
Self loops == 0 is as it should be! 

#### Looking at the ones with no edges

In [51]:
# workflows with a .ga but no pairs: one tool only, or something missed?
no_pairs = galaxy[galaxy["has_ga"] & ~galaxy["id"].isin(pairs["workflow_id"])]
print(len(no_pairs))

# printing the links to each so I cancheck them out
ids = no_pairs["id"].head(10).to_dict().values()
for wf_id in ids:
    print(f"https://workflowhub.eu/workflows/{wf_id}?version=1")

no_pairs[["id", "title"]].head(10)

150
https://workflowhub.eu/workflows/16?version=1
https://workflowhub.eu/workflows/22?version=1
https://workflowhub.eu/workflows/23?version=1
https://workflowhub.eu/workflows/24?version=1
https://workflowhub.eu/workflows/25?version=1
https://workflowhub.eu/workflows/123?version=1
https://workflowhub.eu/workflows/137?version=1
https://workflowhub.eu/workflows/189?version=1
https://workflowhub.eu/workflows/193?version=1
https://workflowhub.eu/workflows/194?version=1


,id,title
14,16,Cheminformatics - TransFS scoring
19,22,Pathway Analysis
20,23,Genetic Design
21,24,RetroSynthesis
22,25,Pathway Ranker
92,123,Compute daily and monthly mean from meteorological stati...
103,137,parallel-accession-download/main
125,189,BioTranslator Workflow
127,193,Workflow of BioTranslator Comparative Analysis
128,194,Protein MD Setup tutorial using BioExcel Building Blocks...


Comments per workflow
```
14    16                            Cheminformatics - TransFS scoring       has a bunch of seemingly non tool nodes
19    22                                             Pathway Analysis       no preview  
20    23                                               Genetic Design       strange - seems to contain things that are in the maping document, also no preview 
21    24                                               RetroSynthesis       many steps no recognised tools, but that should not matter
22    25                                               Pathway Ranker       -"-
92   123  Compute daily and monthly mean from meteorological stati...       -"-
103  137                             parallel-accession-download/main       -"-
125  189                                       BioTranslator Workflow       -"-
127  193               Workflow of BioTranslator Comparative Analysis       only one tool
128  194  Protein MD Setup tutorial using BioExcel Building Blocks...       many steps no recognised tools, but that should not matter
```

I dont know what the conclusion was - maybe need to look into how the files look more. Currently only looked at the overview with the step names and visualisations. I dont feel like that right now though soo TODO

